In [2]:
"""import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')
"""

"import pandas as pd\nimport numpy as np\nfrom sklearn.model_selection import KFold\nfrom sklearn.preprocessing import LabelEncoder\nfrom sklearn.metrics import mean_squared_error\nfrom sklearn.linear_model import Ridge\nimport lightgbm as lgb\nimport xgboost as xgb\nfrom catboost import CatBoostRegressor\nimport warnings\nwarnings.filterwarnings('ignore')\n"

In [3]:
import pandas as pd
import numpy as np

# ============================================================
# 1. VERİYİ YÜKLEYELİM
# ============================================================
train = pd.read_csv('../data/raw/train.csv')
test  = pd.read_csv('../data/raw/test_x.csv')

print("Train shape:", train.shape)
print("Test shape :", test.shape)

TARGET = 'bilissel_performans_skoru'
test_ids = test['id'].copy()

# ============================================================
# 2. TEMİZLİK VE FORMATLAMA (Her iki sete de uygulanabilir adımlar)
# ============================================================
country_map = {
    'Spain': 'Ispanya',
    'South Korea': 'Guney Kore',
    'Sweden': 'Isvec',
    'Mexico': 'Meksika', 
    'Netherlands': 'Hollanda'
}

def basic_clean(df):
    df = df.copy()
    # Ülke çevirisi
    if 'ulke' in df.columns:
        df['ulke'] = df['ulke'].replace(country_map)
    # String temizliği
    df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
    # Gizli NaN değerleri düzeltme
    df = df.replace(['nan', 'NaN', '?', 'None', '', 'null', 'NULL'], np.nan)
    # ID kolonunu düşür
    if 'id' in df.columns:
        df = df.drop('id', axis=1)
    return df

train = basic_clean(train)
test = basic_clean(test)

# ============================================================
# 3. İSTATİSTİKSEL DOLGULAR (Sadece Train'den öğrenip Test'e uyguluyoruz)
# ============================================================

# --- Kategorik Dolgular ---
train['meslek'] = train['meslek'].fillna('Bilinmiyor')
test['meslek']  = test['meslek'].fillna('Bilinmiyor')

train['ruh_sagligi_durumu'] = train['ruh_sagligi_durumu'].fillna('Bilinmiyor')
test['ruh_sagligi_durumu']  = test['ruh_sagligi_durumu'].fillna('Bilinmiyor')

# Kronotip Mode (En çok tekrar eden)
kronotip_mode = train['kronotip'].mode()[0]
train['kronotip'] = train['kronotip'].fillna(kronotip_mode)
test['kronotip']  = test['kronotip'].fillna(kronotip_mode)

# --- Sayısal Dolgular (Median) ---
bmi_median = train['vucut_kitle_indeksi'].median()
train['vucut_kitle_indeksi'] = train['vucut_kitle_indeksi'].fillna(bmi_median)
test['vucut_kitle_indeksi']  = test['vucut_kitle_indeksi'].fillna(bmi_median)

# Kafein (Senin kararın üzerine 0)
train['uyku_oncesi_kafein_mg'] = train['uyku_oncesi_kafein_mg'].fillna(0)
test['uyku_oncesi_kafein_mg']  = test['uyku_oncesi_kafein_mg'].fillna(0)

# Stres Skoru (Gruplandırılmış Median)
stres_map = train.groupby('ruh_sagligi_durumu')['stres_skoru'].median().to_dict()
global_stres_median = train['stres_skoru'].median()

train['stres_skoru'] = train['stres_skoru'].fillna(train['ruh_sagligi_durumu'].map(stres_map))
test['stres_skoru']  = test['stres_skoru'].fillna(test['ruh_sagligi_durumu'].map(stres_map))

# Hâlâ dolmayan varsa global median
train['stres_skoru'] = train['stres_skoru'].fillna(global_stres_median)
test['stres_skoru']  = test['stres_skoru'].fillna(global_stres_median)

# ============================================================
# 4. OUTLIER CAPPING (Sınırları Train belirler)
# ============================================================
outlier_cols = ['vucut_kitle_indeksi', 'uyku_oncesi_kafein_mg', 
                'gunluk_adim_sayisi', 'sekerleme_suresi_dk',
                'gunluk_calisma_saati', 'uykuya_dalma_suresi_dk']

for col in outlier_cols:
    if col in train.columns:
        lower = train[col].quantile(0.01)
        upper = train[col].quantile(0.99)
        
        train[col] = train[col].clip(lower=lower, upper=upper)
        test[col]  = test[col].clip(lower=lower, upper=upper) # Test'e train sınırlarını uygula

print("Pipeline sızıntısız şekilde tamamlandı!")
print(f"Train NaN: {train.isnull().sum().sum()} | Test NaN: {test.isnull().sum().sum()}")

Train shape: (56000, 24)
Test shape : (24000, 23)
Pipeline sızıntısız şekilde tamamlandı!
Train NaN: 0 | Test NaN: 0
